# Preprocessing: PeMS08 traffic data

This notebook turns raw traffic-sensor readings into a labelled dataset for training classifiers
(ANN, CNN, LSTM and so on).

**The task each sample represents:** *given the last 2 hours of readings from one sensor, will the traffic
flow at that sensor in the next 5 minutes be Low, Medium or High?*

| Step | What happens |
|---|---|
| 1 | Load the raw PeMS08 data |
| 2 | Cut sliding 2-hour windows out of each sensor's time series |
| 3 | Split by time: first 80 % of the timeline → train, last 20 % → test |
| 4 | Label each window Low / Medium / High from that sensor's own flow levels |
| 5 | Turn every (window, sensor) pair into one sample |
| 6 | Standardise each sensor's features with its own statistics |
| 7 | Save everything to `traffic_flow_preprocessed.npz` |

Data never leaks from test into training: the split, the label thresholds and the scaling all use training-period data only.
Both the thresholds and the scaling are **per sensor**, because traffic levels differ hugely between detectors.

## 0 · Setup

`sliding_window_view` builds every window at once, without a Python loop.
The settings are grouped here so they're easy to change.

In [1]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import os

# Paths & Config
PROJECT = os.path.abspath(".")  # run the notebooks from the repository root

DATA_DIR = os.path.join(PROJECT, "data")
os.makedirs(DATA_DIR, exist_ok=True)

DATA_FILE = os.path.join(DATA_DIR, "pems08.npz")
OUT_FILE  = os.path.join(DATA_DIR, "traffic_flow_preprocessed.npz")

WINDOW = 24          # 24 steps x 5 min = 2 hours of history
TRAIN_FRAC = 0.8     # chronological split: first 80% of time for training
FLOW = 0             # feature order in PeMS08: 0 = flow, 1 = occupancy, 2 = speed

## 1 · Load the raw data

PeMS08 comes from 170 loop detectors on San Bernardino freeways (California, July–August 2016).
Every 5 minutes, each detector records three features:

| Index | Feature | Meaning | Unit |
|---|---|---|---|
| 0 | flow | vehicles that passed the detector | vehicles / 5 min |
| 1 | occupancy | fraction of the 5 minutes a vehicle was over the detector | 0–1 |
| 2 | speed | average speed of those vehicles | mph |

The array has shape **(timesteps, sensors, features)**. It's cast to `float32`, which halves memory and loses no precision that matters here.

In [2]:
traffic_data = np.load(DATA_FILE)["data"].astype(np.float32)   # (time_steps, sensors, features)
time_steps, num_sensors, num_features = traffic_data.shape
print(f"Data shape: {traffic_data.shape} -> {time_steps} timesteps, "
      f"{num_sensors} sensors, {num_features} features")
print(f"That is {time_steps / 288:.0f} days of 5-minute readings (288 readings per day).")

Data shape: (17856, 170, 3) -> 17856 timesteps, 170 sensors, 3 features
That is 62 days of 5-minute readings (288 readings per day).


## 2 · Cut sliding windows

A **window** is 24 consecutive readings (2 hours) from one sensor. The window slides forward one step
at a time, so window *i* covers timesteps *i … i+23*. Its **target** is the flow at the next step, *i+24*.

```
time →   t0 t1 t2 … t23 | t24
         └──── input ───┘  target (flow)
            t1 t2 … t23 t24 | t25          ← next window, shifted by one step
```

`traffic_data[:-1]` drops the final timestep, so every window still has a "next step" to predict.
Result: `windows` has shape **(n_windows, sensors, 24, features)**.

In [3]:
# windows[i, s] = traffic_data[i : i + WINDOW, s, :]  ->  (n_windows, sensors, WINDOW, features)
windows = sliding_window_view(traffic_data[:-1], WINDOW, axis=0).transpose(0, 1, 3, 2)
next_flow = traffic_data[WINDOW:, :, FLOW]                    # (n_windows, sensors): flow at t + WINDOW
n_windows = windows.shape[0]                                  # time_steps - WINDOW

print("windows  :", windows.shape, " (n_windows, sensors, window, features)")
print("next_flow:", next_flow.shape, " (n_windows, sensors)")

windows  : (17832, 170, 24, 3)  (n_windows, sensors, window, features)
next_flow: (17832, 170)  (n_windows, sensors)


## 3 · Split by time, not at random

Neighbouring windows share 23 of their 24 readings, so they're almost identical. A **random** split would put
near-copies of test samples into training, and the model would score well by memorising rather than predicting.

The split is therefore **chronological**: windows from the first 80 % of the timeline are for training,
the rest are for testing. This matches real use, where a model trained on the past predicts the future.

`train_time_end` is the last timestep any training window touches. Everything fitted later in this notebook
(thresholds, scaling) uses only data before it.

In [4]:
split = int(TRAIN_FRAC * n_windows)                           # windows before this index -> train
train_time_end = split + WINDOW                               # last timestep seen by any training window

print(f"Train windows: 0 … {split - 1}   ({split} per sensor)")
print(f"Test windows : {split} … {n_windows - 1}   ({n_windows - split} per sensor)")
print(f"Training period covers timesteps 0 … {train_time_end - 1} (≈ day {train_time_end / 288:.1f})")

Train windows: 0 … 14264   (14265 per sensor)
Test windows : 14265 … 17831   (3567 per sensor)
Training period covers timesteps 0 … 14288 (≈ day 49.6)


## 4 · Label each window Low / Medium / High

Sensors differ a lot. A busy freeway detector might see 500 vehicles per 5 minutes, a quiet ramp 50. So each
sensor gets **its own** thresholds, from its flow distribution during the training period:

| Class | Rule (per sensor) |
|---|---|
| 0 · Low | next-step flow ≤ 33rd percentile |
| 1 · Medium | 33rd percentile < flow ≤ 66th percentile |
| 2 · High | flow > 66th percentile |

With percentile cut-offs, the three classes come out about equally sized. `np.where` labels every window of every sensor at once.

In [5]:
low_th = np.percentile(traffic_data[:train_time_end, :, FLOW], 33, axis=0)    # (sensors,)
high_th = np.percentile(traffic_data[:train_time_end, :, FLOW], 66, axis=0)
labels = np.where(next_flow <= low_th, 0, np.where(next_flow <= high_th, 1, 2)).astype(np.int8)

print("Example thresholds (vehicles / 5 min):")
for s in [0, 50, 169]:
    print(f"  sensor {s:>3}: Low ≤ {low_th[s]:.0f} < Medium ≤ {high_th[s]:.0f} < High")

Example thresholds (vehicles / 5 min):
  sensor   0: Low ≤ 281 < Medium ≤ 422 < High
  sensor  50: Low ≤ 153 < Medium ≤ 240 < High
  sensor 169: Low ≤ 15 < Medium ≤ 35 < High


## 5 · One sample per (window, sensor) pair

So far the data is a grid of *windows × sensors*. Models expect a flat list of samples, so each
(window, sensor) pair becomes one sample of shape **(24, 3)**.

Two small bookkeeping arrays go with each sample:

* `sensor_id` - which sensor it came from
* `time_idx` - the timestep where its window starts

With these you can trace any prediction back to the raw data, or evaluate per sensor later.

In [6]:
def to_samples(w, y, t_offset):
    n, s = y.shape
    X = w.reshape(n * s, WINDOW, num_features)
    sensor_id = np.tile(np.arange(s, dtype=np.int16), n)
    time_idx = np.repeat(np.arange(t_offset, t_offset + n, dtype=np.int32), s)
    return X, y.reshape(-1), sensor_id, time_idx

X_train_seq, y_train, sensor_train, time_train = to_samples(windows[:split], labels[:split], 0)
X_test_seq, y_test, sensor_test, time_test = to_samples(windows[split:], labels[split:], split)

print("X_train_seq:", X_train_seq.shape, " X_test_seq:", X_test_seq.shape)

X_train_seq: (2425050, 24, 3)  X_test_seq: (606390, 24, 3)


## 6 · Standardise each sensor with its own statistics

Two scale problems need fixing:

1. **Features differ in scale.** Flow is in the hundreds, occupancy below 1, speed around 60–70.
2. **Sensors differ in scale.** A busy freeway detector averages about 400 vehicles per 5 minutes, a quiet ramp about 20.

The second one matters because the labels are **per sensor**. A flow of 200 is *High* on the ramp but *Low* on
the freeway. With one global scaling, a model sees the same input with different correct answers, and it can't
tell the sensors apart. (In `baselines.ipynb`, logistic regression scores 69 % with global scaling and 87 % with per-sensor scaling.)

So every sensor *s* and feature *f* is standardised with that sensor's own mean and standard deviation:

$$x'_{s,f} = \frac{x_{s,f} - \mu_{s,f}}{\sigma_{s,f}}$$

After this, "+1" means "one standard deviation busier than **this sensor's** normal", which lines up with the
per-sensor labels.

μ and σ come from the **training period only** (timesteps before `train_time_end`). The test set uses those same
numbers, as if it were unseen future data.

In [7]:
train_period = traffic_data[:train_time_end]                              # (timesteps, sensors, features)
mean = train_period.mean(axis=0, dtype=np.float64).astype(np.float32)     # (sensors, features)
std = (train_period.std(axis=0, dtype=np.float64) + 1e-6).astype(np.float32)

# Each sample uses the statistics of its own sensor; [:, None, :] broadcasts over the 24 timesteps.
X_train_seq = (X_train_seq - mean[sensor_train][:, None, :]) / std[sensor_train][:, None, :]
X_test_seq = (X_test_seq - mean[sensor_test][:, None, :]) / std[sensor_test][:, None, :]

print("Per-sensor statistics, examples (training period):")
print(f"  {'sensor':>6} | {'flow mean':>9} {'flow std':>8} | {'occ mean':>8} {'occ std':>7} | {'speed mean':>10} {'speed std':>9}")
for s in [0, 50, 169]:
    (fm, om, sm), (fs, os_, ss) = mean[s], std[s]
    print(f"  {s:>6} | {fm:9.1f} {fs:8.1f} | {om:8.3f} {os_:7.3f} | {sm:10.1f} {ss:9.1f}")

Per-sensor statistics, examples (training period):
  sensor | flow mean flow std | occ mean occ std | speed mean speed std
       0 |     332.7    147.5 |    0.112   0.051 |       62.8       5.2
      50 |     183.7     84.9 |    0.064   0.033 |       65.6       3.1
     169 |      25.6     18.0 |    0.016   0.034 |       64.2       5.5


## Check the result

The class shares should all be about a third in both splits. The spot check un-scales one sample and compares it
with the raw data, to confirm windows and labels line up correctly.

In [8]:
print(f"Dataset ready: train {X_train_seq.shape}, test {X_test_seq.shape}")
for name, y in [("train", y_train), ("test", y_test)]:
    share = np.bincount(y, minlength=3) / len(y)
    print(f"  {name} class share  Low {share[0]:.3f}  Medium {share[1]:.3f}  High {share[2]:.3f}")

# Spot check: sensor 5, window starting at timestep 100
i = np.where((sensor_train == 5) & (time_train == 100))[0][0]
restored = X_train_seq[i] * std[5] + mean[5]
print("\nSpot check (sensor 5, t = 100)")
print("  window matches raw data:", np.allclose(restored, traffic_data[100:100 + WINDOW, 5], atol=1e-2))
print("  label:", ["Low", "Medium", "High"][y_train[i]],
      f"(next flow = {traffic_data[100 + WINDOW, 5, FLOW]:.0f})")

Dataset ready: train (2425050, 24, 3), test (606390, 24, 3)
  train class share  Low 0.331  Medium 0.332  High 0.337
  test class share  Low 0.337  Medium 0.327  High 0.336

Spot check (sensor 5, t = 100)
  window matches raw data: True
  label: High (next flow = 326)


## 7 · Save

Only the sequence format `(samples, 24, 3)` is saved. Dense models (ANN) need a flat vector,
which is just a reshape, no need to store a second copy:

```python
X_flat = X_seq.reshape(len(X_seq), -1)      # (samples, 72)
```

| Key | Shape | Contents |
|---|---|---|
| `X_train_seq`, `X_test_seq` | (samples, 24, 3) | standardised input windows |
| `y_train`, `y_test` | (samples,) | class labels 0 / 1 / 2 |
| `sensor_train`, `sensor_test` | (samples,) | sensor each sample came from |
| `time_train`, `time_test` | (samples,) | window start timestep |
| `feature_mean`, `feature_std` | (170, 3) | per-sensor scaling values: row *s* is sensor *s* (needed to undo the scaling or scale new data) |
| `low_th`, `high_th` | (170,) | per-sensor class thresholds |
| `window` | scalar | window length (24) |

In [9]:
np.savez(OUT_FILE,
         X_train_seq=X_train_seq, X_test_seq=X_test_seq,
         y_train=y_train, y_test=y_test,
         sensor_train=sensor_train, sensor_test=sensor_test,
         time_train=time_train, time_test=time_test,
         feature_mean=mean, feature_std=std,
         low_th=low_th, high_th=high_th, window=WINDOW)
print(f"Saved {OUT_FILE}")

Saved E:\Traffic Flow Detection Using Neural Networks\data\traffic_flow_preprocessed.npz
